# ⚔️ Adversarial Attacks on EuroSAT Satellite Dataset
### Testing Solutions Internship | AI Testing
**Dataset:** EuroSAT (10 land-use classes from Sentinel-2 satellite images)
**White-Box Attack:** FGSM (Fast Gradient Sign Method)
**Black-Box Attack:** Boundary Attack

---
| Part | Task |
|------|------|
| A | Load EuroSAT dataset + train CNN model |
| B | White-Box Attack — FGSM |
| C | Black-Box Attack — Boundary Attack |
| D | Results, comparison, and visualization |


## ⚙️ Step 1 — Install & Import

In [ ]:
!pip install tensorflow-datasets foolbox -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
import foolbox as fb
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries ready!")
print(f"   TensorFlow : {tf.__version__}")
print(f"   Foolbox    : {fb.__version__}")


## 📂 Step 2 — Load EuroSAT Dataset
EuroSAT has **27,000 satellite images** across **10 land-use classes**.
No Kaggle token needed — downloads automatically!


In [ ]:
# Load EuroSAT via TensorFlow Datasets
print("Downloading EuroSAT dataset (this takes ~1-2 minutes)...")

(ds_train, ds_test), info = tfds.load(
    'eurosat/rgb',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,
    with_info=True
)

CLASS_NAMES = info.features['label'].names
NUM_CLASSES = len(CLASS_NAMES)
IMG_SIZE    = 64

print(f"\n✅ EuroSAT loaded!")
print(f"   Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"   Train size : {tf.data.experimental.cardinality(ds_train).numpy()}")
print(f"   Test size  : {tf.data.experimental.cardinality(ds_test).numpy()}")


In [ ]:
# Preprocess
def preprocess(image, label):
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

BATCH_SIZE = 32

train_ds = ds_train.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds  = ds_test.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Preview sample images
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle("EuroSAT Dataset — Sample Images (10 Classes)", fontsize=14, fontweight='bold')

for images, labels in ds_train.map(preprocess).batch(50).take(1):
    seen = set()
    idx  = 0
    for i in range(len(labels)):
        lbl = labels[i].numpy()
        if lbl not in seen:
            seen.add(lbl)
            ax = axes[idx // 5][idx % 5]
            ax.imshow(images[i].numpy())
            ax.set_title(CLASS_NAMES[lbl], fontsize=9)
            ax.axis('off')
            idx += 1
            if idx == 10: break

plt.tight_layout()
plt.show()


## 🧠 Step 3 — Train CNN Model (TensorFlow)
Train a compact CNN on EuroSAT. This model will be the **target** for both attacks.


In [ ]:
# Build CNN model
model = keras.Sequential([
    keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation='softmax')
], name="EuroSAT_CNN")

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# Train
print("Training CNN on EuroSAT...")
history = model.fit(train_ds, epochs=10, validation_data=test_ds, verbose=1)

loss, acc = model.evaluate(test_ds, verbose=0)
print(f"\n✅ Model trained!")
print(f"   Test Accuracy : {acc*100:.2f}%")
print(f"   Test Loss     : {loss:.4f}")


In [ ]:
# Plot training
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("EuroSAT CNN — Training History", fontsize=13, fontweight='bold')

axes[0].plot(history.history['accuracy'],     label='Train', color='#2196F3')
axes[0].plot(history.history['val_accuracy'], label='Val',   color='#4CAF50')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train', color='#F44336')
axes[1].plot(history.history['val_loss'], label='Val',   color='#FF9800')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
model.save("eurosat_model.keras")
print("✅ Model saved!")


---
# ⚔️ PART B — White-Box Attack: FGSM
## What is FGSM?
FGSM adds a small perturbation in the direction of the gradient of the loss w.r.t the input.
The attacker needs **full access to model gradients** — that's why it's white-box.

**Formula:** `x_adv = x + epsilon × sign(∇x Loss(model(x), y))`


In [ ]:
def fgsm_attack(model, images, labels, epsilon):
    """
    Fast Gradient Sign Method (White-Box Attack).
    Perturbs images in direction that maximizes model loss.
    """
    images = tf.cast(images, tf.float32)
    labels = tf.cast(labels, tf.int64)

    with tf.GradientTape() as tape:
        tape.watch(images)
        predictions = model(images, training=False)
        loss = keras.losses.sparse_categorical_crossentropy(labels, predictions)

    # Get gradient of loss w.r.t. input image
    gradients  = tape.gradient(loss, images)
    # Add perturbation in gradient sign direction
    perturbation = epsilon * tf.sign(gradients)
    adv_images   = images + perturbation
    # Clip to valid pixel range [0, 1]
    adv_images   = tf.clip_by_value(adv_images, 0, 1)
    return adv_images

print("✅ FGSM attack function defined!")


In [ ]:
# Run FGSM with different epsilon values
epsilons = [0.0, 0.01, 0.05, 0.1, 0.2, 0.3]
fgsm_accuracies = []

# Get test batch
test_images, test_labels = [], []
for imgs, lbls in test_ds.take(10):
    test_images.append(imgs)
    test_labels.append(lbls)
test_images = tf.concat(test_images, axis=0)[:200]
test_labels = tf.concat(test_labels, axis=0)[:200]

print("Running FGSM at different epsilon values...\n")
for eps in epsilons:
    adv_imgs = fgsm_attack(model, test_images, test_labels, eps)
    preds    = model(adv_imgs, training=False)
    acc      = tf.reduce_mean(
        tf.cast(tf.argmax(preds, axis=1) == tf.cast(test_labels, tf.int64), tf.float32)
    ).numpy()
    fgsm_accuracies.append(acc * 100)
    print(f"  Epsilon {eps:.2f} → Model Accuracy: {acc*100:.1f}%  Attack Success: {100 - acc*100:.1f}%")


In [ ]:
# Visualize FGSM results on sample images
sample_imgs  = test_images[:5]
sample_lbls  = test_labels[:5]

fig, axes = plt.subplots(5, len(epsilons), figsize=(18, 15))
fig.suptitle("FGSM White-Box Attack — Original vs Adversarial Images", fontsize=14, fontweight='bold')

for col, eps in enumerate(epsilons):
    adv_imgs = fgsm_attack(model, sample_imgs, sample_lbls, eps)
    for row in range(5):
        ax  = axes[row][col]
        img = adv_imgs[row].numpy()
        pred = CLASS_NAMES[tf.argmax(model(adv_imgs[row:row+1]), axis=1).numpy()[0]]
        true = CLASS_NAMES[sample_lbls[row].numpy()]
        correct = pred == true
        ax.imshow(img)
        ax.set_title(f"ε={eps}\n{pred}", fontsize=7,
                     color='green' if correct else 'red')
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(f"True:\n{true}", fontsize=7, rotation=0, labelpad=50, va='center')

plt.tight_layout()
plt.savefig("fgsm_results.png", dpi=120, bbox_inches='tight')
plt.show()
print("✅ FGSM visualization saved!")


---
# 🕵️ PART C — Black-Box Attack: Boundary Attack
## What is Boundary Attack?
The Boundary Attack has **NO access to model gradients or weights**.
It only needs to **query the model and observe the predicted class**.

**How it works:**
1. Start from an image already misclassified (e.g., pure noise that happens to be wrong class)
2. Walk along the decision boundary toward the original image
3. Stay adversarial (wrong class) while getting closer to the original
4. Result: adversarial image very close to original but still misclassified

This simulates attacking a deployed API where you can only see the output label.


In [ ]:
# Convert TF model to Foolbox model for black-box attack
# Foolbox wraps the model and treats it as a black box (only uses predictions)

import foolbox as fb
import torch

# Create a Foolbox TensorFlow model wrapper
fmodel = fb.TensorFlowModel(model, bounds=(0, 1))

# Get sample images for black-box attack (small batch for speed)
bb_images, bb_labels = [], []
for imgs, lbls in test_ds.take(2):
    bb_images.append(imgs[:5])
    bb_labels.append(lbls[:5])
    break

bb_images = tf.concat(bb_images, axis=0)
bb_labels = tf.concat(bb_labels, axis=0)

print(f"✅ Foolbox model ready!")
print(f"   Testing on {len(bb_images)} images")
print(f"   True labels: {[CLASS_NAMES[l] for l in bb_labels.numpy()]}")

# Check clean accuracy on these samples
clean_preds = model(bb_images, training=False)
clean_acc = tf.reduce_mean(
    tf.cast(tf.argmax(clean_preds,1) == tf.cast(bb_labels, tf.int64), tf.float32)
).numpy()
print(f"   Clean accuracy on sample: {clean_acc*100:.1f}%")


In [ ]:
# Run Boundary Attack (Black-Box)
print("Running Boundary Attack (Black-Box)...")
print("Note: This attack only queries the model output — no gradients used!\n")

# Boundary Attack via Foolbox
attack = fb.attacks.BoundaryAttack()

bb_images_np = bb_images.numpy()
bb_labels_np = bb_labels.numpy()

# Run attack
raw, clipped, is_adv = attack(
    fmodel,
    tf.constant(bb_images_np, dtype=tf.float32),
    tf.constant(bb_labels_np, dtype=tf.int32),
    epsilons=None
)

adv_images_bb = clipped.numpy() if hasattr(clipped, 'numpy') else np.array(clipped)

# Evaluate
bb_preds  = model(adv_images_bb, training=False)
bb_acc    = tf.reduce_mean(
    tf.cast(tf.argmax(bb_preds,1) == tf.cast(bb_labels, tf.int64), tf.float32)
).numpy()

print(f"\n✅ Boundary Attack complete!")
print(f"   Clean accuracy   : {clean_acc*100:.1f}%")
print(f"   After BB attack  : {bb_acc*100:.1f}%")
print(f"   Attack success   : {(1-bb_acc)*100:.1f}%")
print(f"   Adversarial?     : {is_adv.numpy()}")


In [ ]:
# Visualize Boundary Attack results
fig, axes = plt.subplots(3, min(5, len(bb_images)), figsize=(16, 10))
fig.suptitle("Boundary Attack (Black-Box) — Original vs Adversarial", fontsize=13, fontweight='bold')

n = min(5, len(bb_images))
for i in range(n):
    orig_img = bb_images[i].numpy()
    adv_img  = adv_images_bb[i]
    diff_img = np.abs(orig_img - adv_img) * 10  # amplify diff for visibility

    true_lbl  = CLASS_NAMES[bb_labels[i].numpy()]
    orig_pred = CLASS_NAMES[tf.argmax(model(bb_images[i:i+1]), 1).numpy()[0]]
    adv_pred  = CLASS_NAMES[tf.argmax(model(adv_img[np.newaxis,...]), 1).numpy()[0]]

    # Row 1: Original
    axes[0][i].imshow(orig_img)
    axes[0][i].set_title(f"Original\n{orig_pred}", fontsize=8,
                          color='green' if orig_pred==true_lbl else 'red')
    axes[0][i].axis('off')

    # Row 2: Adversarial
    axes[1][i].imshow(np.clip(adv_img, 0, 1))
    axes[1][i].set_title(f"Adversarial\n{adv_pred}", fontsize=8,
                          color='green' if adv_pred==true_lbl else 'red')
    axes[1][i].axis('off')

    # Row 3: Difference (amplified)
    axes[2][i].imshow(np.clip(diff_img, 0, 1), cmap='hot')
    axes[2][i].set_title("Perturbation\n(amplified 10x)", fontsize=8)
    axes[2][i].axis('off')

axes[0][0].set_ylabel("Original", fontsize=9, fontweight='bold')
axes[1][0].set_ylabel("Adversarial", fontsize=9, fontweight='bold')
axes[2][0].set_ylabel("Difference", fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig("boundary_attack_results.png", dpi=120, bbox_inches='tight')
plt.show()
print("✅ Boundary attack visualization saved!")


---
# 📊 PART D — Comparison & Final Results


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Adversarial Attacks on EuroSAT — Final Results", fontsize=14, fontweight='bold')

# 1. FGSM accuracy vs epsilon
axes[0].plot(epsilons, fgsm_accuracies, 'o-', color='#F44336', linewidth=2, markersize=8)
axes[0].fill_between(epsilons, fgsm_accuracies, alpha=0.1, color='#F44336')
axes[0].set_title('FGSM (White-Box)\nModel Accuracy vs Epsilon')
axes[0].set_xlabel('Epsilon (perturbation strength)')
axes[0].set_ylabel('Model Accuracy (%)')
axes[0].grid(True, alpha=0.3)
for x, y in zip(epsilons, fgsm_accuracies):
    axes[0].annotate(f'{y:.1f}%', (x, y), textcoords="offset points",
                     xytext=(0, 10), ha='center', fontsize=8)

# 2. Attack comparison bar
attack_names   = ['Clean', 'FGSM\n(ε=0.1)', 'FGSM\n(ε=0.2)', 'Boundary\nAttack']
attack_acc     = [fgsm_accuracies[0], fgsm_accuracies[3], fgsm_accuracies[4], bb_acc*100]
bar_colors     = ['#4CAF50', '#FF9800', '#F44336', '#9C27B0']

bars = axes[1].bar(attack_names, attack_acc, color=bar_colors)
axes[1].set_title('Attack Comparison\nModel Accuracy After Attack')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(0, 110)
for bar, val in zip(bars, attack_acc):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+1,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=9)

# 3. Attack success rate
success_rates = [100 - a for a in attack_acc]
axes[2].bar(attack_names, success_rates, color=bar_colors)
axes[2].set_title('Attack Success Rate\n(higher = more dangerous)')
axes[2].set_ylabel('Attack Success Rate (%)')
axes[2].set_ylim(0, 110)
for i, val in enumerate(success_rates):
    axes[2].text(i, val+1, f'{val:.1f}%', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig("attack_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Comparison charts saved!")


## ✅ Summary

### What We Implemented:
| | White-Box (FGSM) | Black-Box (Boundary Attack) |
|---|---|---|
| **Access needed** | Model gradients | Output label only |
| **How it works** | Gradient sign perturbation | Walk along decision boundary |
| **Speed** | Very fast (1 step) | Slow (many queries) |
| **Dataset** | EuroSAT (10 classes) | EuroSAT (10 classes) |
| **Framework** | TensorFlow | Foolbox (wraps TF model) |

### How to Explain to Mentor:
> *"I implemented two adversarial attacks on the EuroSAT satellite dataset. For white-box, I used FGSM which uses model gradients to add perturbations — as epsilon increases, model accuracy drops significantly. For black-box, I used the Boundary Attack via Foolbox which only queries the model output without any gradient access, simulating an attack on a deployed API. Both attacks were applied to a CNN trained on EuroSAT and the results were compared visually and quantitatively."*
